# Bhutan GST Version-Aware RAG: Full Progress to Controlled Evaluation

This notebook combines the earlier mid-review work with the next proposal stages. It covers corpus completion, version metadata, MiniLM results, three controlled retrieval systems, the 120-question benchmark, generator selection, and the provisional controlled comparison.

**Research warning:** automatically extracted relationships, generated benchmark questions, and answer scores still require group review. The prototype is not legal advice.

## 1. Proposal alignment

The proposal requires: (1) an official versioned corpus, (2) MiniLM fine-tuning, (3) standard, date-filtered, and version-aware RAG, (4) a 120-question benchmark, and (5) controlled evaluation. The frontend and API are intentionally outside this notebook.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

from gst_rag.paths import BENCHMARK, INDEXES, MANIFEST, METADATA, PROCESSED, RESULTS, TRAINING
from gst_rag.utils import read_jsonl

pd.set_option('display.max_colwidth', 100)
print('Project root:', Path.cwd())

## 2. Previous mid-review progress

The earlier checkpoint collected official PDFs, extracted legal sections, generated training triplets, fine-tuned MiniLM, and built BM25 plus FAISS hybrid retrieval. The cells below read the saved artifacts instead of inventing new counts.

In [ ]:
manifest = pd.read_csv(MANIFEST).fillna('')
sections = read_jsonl(PROCESSED / 'sections.jsonl')
training = read_jsonl(TRAINING / 'minilm_training.jsonl')
events = pd.read_csv(METADATA / 'amendment_events.csv').fillna('')
extraction = pd.read_csv(METADATA / 'extraction_report.csv').fillna('')

progress = pd.DataFrame({
    'Item': ['Manifest records', 'Available PDFs', 'Usable documents', 'Searchable chunks', 'Training triplets', 'Amendment events'],
    'Count': [
        len(manifest),
        int((manifest.availability_status == 'available').sum()),
        int(extraction.status.astype(str).str.startswith('extracted').sum()),
        len(sections),
        len(training),
        len(events),
    ]
})
display(progress)

## 3. Corpus completion and preprocessing audit

The missing official 2026 Amendment Act is now included. The scanned December 2025 notice uses a visual transcription fallback because the PDF contains no text layer. Its transcription remains flagged for group review.

In [ ]:
display(manifest[[
    'document_id', 'title', 'document_type', 'effective_from',
    'availability_status', 'official_status'
]])
display(extraction[['document_id', 'status', 'pages', 'english_pages', 'characters']])

## 4. Version and amendment metadata

Each chunk now includes a version family, effective interval, version rank, and links to detected amendment events. `source_text_verified_needs_group_review` means the relationship is visible in the official source text, but the group must still complete legal verification.

In [ ]:
print(events.verification_status.value_counts())
display(events[[
    'event_id', 'source_document_id', 'affected_section', 'relation_type',
    'effective_from', 'verification_status'
]].head(12))
version_fields = ['chunk_id', 'version_family', 'version_start', 'version_end', 'version_status', 'affected_sections']
section_frame = pd.DataFrame(sections)
display(section_frame.loc[section_frame.amendment_event_ids.map(len) > 0, version_fields].head(8))

## 5. MiniLM fine-tuning result from the mid-review

These are development-set retrieval results, not the final 120-question benchmark. The fine-tuned model was selected because both Recall@5 and MRR@10 improved.

In [ ]:
minilm = json.loads((RESULTS / 'minilm_metrics.json').read_text(encoding='utf-8'))
display(pd.DataFrame([
    {'Model': 'Pretrained MiniLM', **minilm['before']},
    {'Model': 'Fine-tuned MiniLM', **minilm['after']},
]))
print('Selected:', minilm['selection_rule_result'])

## 6. Three controlled retrieval systems

All systems share the same corpus, chunks, fine-tuned MiniLM embeddings, BM25 retrieval, FAISS index, and RRF fusion. Only version handling changes:

- **Standard:** no date or version filtering.
- **Date-filtered:** retains chunks valid on the target date.
- **Version-aware:** applies dates plus version families and amendment/repeal relationships.

In [ ]:
RUN_LIVE_RETRIEVAL_DEMO = False
if RUN_LIVE_RETRIEVAL_DEMO:
    from gst_rag.systems import RetrievalEngine
    engine = RetrievalEngine.from_disk()
    demo_question = 'As of 2026-01-01, how was GST Act Section 40 amended?'
    demo_rows = []
    for system in ['standard', 'date_filtered', 'version_aware']:
        result = engine.search(demo_question, system=system, target_date='2026-01-01', top_k=1)[0]
        demo_rows.append({'System': system, 'Top document': result['document_id'], 'Section': result['section_number'], 'Page': result['page_number']})
    display(pd.DataFrame(demo_rows))
else:
    saved_predictions = pd.read_csv(RESULTS / 'controlled_comparison_predictions.csv')
    sample_id = saved_predictions.query("category == 'amendment_sensitive'").question_id.iloc[0]
    display(saved_predictions.loc[saved_predictions.question_id == sample_id, ['system', 'question', 'top_document_id', 'top_section', 'gold_document_id', 'gold_section']])
    print('Set RUN_LIVE_RETRIEVAL_DEMO=True to run the search engine instead of reading saved results.')

## 7. Proposal-shaped 120-question benchmark

The benchmark follows the proposed category counts and uses a stratified 20-question development split plus 100 held-out test questions. The questions are source-grounded but remain marked for group review; they must not be described as manually verified final research data yet.

In [ ]:
benchmark = read_jsonl(BENCHMARK / 'gst_120_benchmark.jsonl')
benchmark_frame = pd.DataFrame(benchmark)
display(pd.crosstab(benchmark_frame.category, benchmark_frame.split, margins=True))
display(benchmark_frame[[
    'question_id', 'split', 'category', 'question', 'target_date',
    'gold_document_id', 'gold_section', 'verification_status'
]].head(10))

## 8. Generator comparison

The proposal requires two pretrained instruction models. FLAN-T5-small and FLAN-T5-base were compared on the 20-question development set with the same evidence-only prompt. FLAN-T5-base was selected using the recorded development answer-overlap proxy.

In [ ]:
generator_comparison = json.loads((RESULTS / 'generator_comparison.json').read_text(encoding='utf-8'))
generator_table = pd.DataFrame([
    {
        'Model': model,
        'Development questions': values['development_questions'],
        'Mean token-F1 proxy': values['mean_answer_token_f1_proxy'],
        'Abstention rate': values['abstention_rate'],
    }
    for model, values in generator_comparison['results'].items()
])
display(generator_table)
print('Selected generator:', generator_comparison['selected_generator'])

## 9. Controlled comparison results

Recall@5, MRR@10, and Version Selection Accuracy are the main retrieval results. Answer token-F1 is only an automated proxy; the proposal's final 0/1/2 Answer Accuracy requires manual scoring by the group.

In [ ]:
comparison = json.loads((RESULTS / 'controlled_comparison_metrics.json').read_text(encoding='utf-8'))
comparison_table = pd.DataFrame([
    {
        'System': system,
        'Recall@5': comparison[system]['recall_at_5'],
        'MRR@10': comparison[system]['mrr_at_10'],
        'Version Selection Accuracy': comparison[system]['version_selection_accuracy'],
        'Citation Accuracy': comparison[system]['citation_accuracy'],
        'Answer token-F1 proxy': comparison[system]['answer_token_f1_proxy'],
        'Abstention rate': comparison[system]['abstention_rate'],
    }
    for system in ['standard', 'date_filtered', 'version_aware']
])
numeric_columns = [column for column in comparison_table.columns if column != 'System']
comparison_table[numeric_columns] = comparison_table[numeric_columns].round(3)
display(comparison_table)

In [ ]:
success = comparison['success_criteria']
display(pd.DataFrame([success]).T.rename(columns={0: 'Result'}))

## 10. Interpretation and remaining human checks

The provisional version-aware system improves substantially over both baselines, but it does not meet the proposed 85% Version Selection Accuracy target. Generator answer quality is also a bottleneck. Before these are final research results, the group must:

1. Review the manual transcription of the scanned notice.
2. Verify all 62 amendment relationships and effective dates.
3. Manually review the 300 training triplets.
4. Manually review all 120 benchmark questions and their gold labels.
5. Fill the manual 0/1/2 answer score and citation-check columns in the prediction CSV.
6. Rerun the final summary after corrections.

## 11. Optional full rebuild

The following cell contains the complete reproducible command sequence. It is disabled by default because model training and generator evaluation are expensive on CPU. Set `RUN_EXPENSIVE_STEPS = True` only when you intentionally want to rebuild every artifact.

In [ ]:
import subprocess
import sys

RUN_EXPENSIVE_STEPS = False
commands = [
    [sys.executable, 'scripts/run_to_mid_review.py', '--skip-download', '--skip-training'],
    [sys.executable, 'scripts/train_minilm.py', '--epochs', '1'],
    [sys.executable, 'scripts/build_indexes.py'],
    [sys.executable, 'scripts/compare_generators.py'],
    [sys.executable, 'scripts/run_generator_system.py', 'standard', '--start', '0', '--end', '60'],
    [sys.executable, 'scripts/run_generator_system.py', 'standard', '--start', '60', '--end', '120'],
    [sys.executable, 'scripts/run_generator_system.py', 'date_filtered', '--start', '0', '--end', '60'],
    [sys.executable, 'scripts/run_generator_system.py', 'date_filtered', '--start', '60', '--end', '120'],
    [sys.executable, 'scripts/run_generator_system.py', 'version_aware', '--start', '0', '--end', '60'],
    [sys.executable, 'scripts/run_generator_system.py', 'version_aware', '--start', '60', '--end', '120'],
    [sys.executable, 'scripts/merge_generator_results.py'],
]
if RUN_EXPENSIVE_STEPS:
    for command in commands:
        print('Running:', ' '.join(command), flush=True)
        subprocess.run(command, check=True)
else:
    print('Expensive rebuild skipped. Existing validated artifacts were used.')